# Notebook 02: Pilot Report

**Purpose**: Analyze cached results and generate the P3 gate decision.

**Expected runtime**: ~5 minutes

**GPU cost**: None (CPU only)

**Important**: This notebook reads the cache in read-only mode. It does not require GPU. Run on CPU after notebook 01 completes.

## Step 1: Bootstrap the code from GitHub

Each Kaggle notebook is its own session and `/kaggle/working` does **not** carry over
between them, so every notebook fetches the code itself. Run this cell first.

**Private repo?** Store a GitHub fine-grained PAT (Contents: Read-only) as a Kaggle
Secret named `GITHUB_TOKEN` (Add-ons -> Secrets), then set `USE_SECRET = True`.
Never paste a token into a notebook cell: saved notebook versions keep their source,
so a pasted token is a published token.


In [ ]:
# ---- EDIT IF NEEDED ----
GITHUB_USER = "yoadjei"
GITHUB_REPO = "shiftprofile"
BRANCH      = "main"
USE_SECRET  = False   # True if the repo is private
# ------------------------

import subprocess, sys
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/shiftprofile")

if USE_SECRET:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    _url = f"https://{_tok}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
else:
    _url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

# NOTE: never print _url - it may embed the token.
_git = ["git", "-C", str(REPO_ROOT)]
if REPO_ROOT.exists():
    subprocess.run(_git + ["fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(_git + ["reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, _url, str(REPO_ROOT)], check=True)
del _url

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT)], check=True)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))   # pip -e may not register in a live kernel

import shiftprofile
_sha = subprocess.run(_git + ["rev-parse", "--short", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
print(f"shiftprofile ready at {REPO_ROOT}")
print(f"commit {_sha} on branch {BRANCH}")
print("Record this commit: every result record stores the code version that produced it.")


## Step 2: Load configuration and cache

In [ ]:
from pathlib import Path
import yaml
from shiftprofile.cache import ArtifactCache

repo_root = Path('/kaggle/working/shiftprofile')
config_path = repo_root / 'configs' / 'pilot.yaml'
cache_write = Path('/kaggle/working/cache')
cache_read = Path('/kaggle/input/shiftprofile-cache')

with open(config_path) as f:
    config = yaml.safe_load(f)

print(f"Configuration loaded from {config_path}")
print(f"\nPilot setup:")
print(f"  Models: {config['models']}")
print(f"  Seeds: {config['seeds']}")
print(f"  Shift families: {config['shift_families']}")
print(f"  Severities: {config['severities']}")
print(f"  Explainers: {config['explainers']}")

if cache_read.exists():
    cache = ArtifactCache(write_root=cache_write, read_roots=[cache_read, cache_write])
    print(f"\nCache initialized (read from {cache_read})")
else:
    cache = ArtifactCache(write_root=cache_write, read_roots=[cache_write])
    print(f"\nCache initialized (read from {cache_write} only)")
    print(f"Warning: cache_read not found. Results may be incomplete.")

## Step 3: Compute per-cell calibration metrics

In [ ]:
import numpy as np
from pathlib import Path
from shiftprofile.cells import Cell, enumerate_cells
from shiftprofile.metrics.calibration import apply_temperature, brier_score, ece_equal_mass, ece_debiased, aurc
from shiftprofile.predict import PREDICT_VERSION
from scipy.special import softmax
import yaml
from shiftprofile.data import load_cifar10_test, fixed_eval_indices

repo_root = Path('/kaggle/working/shiftprofile')
config_path = repo_root / 'configs' / 'pilot.yaml'
data_root = Path('/kaggle/working/data')

with open(config_path) as f:
    config = yaml.safe_load(f)

indices = fixed_eval_indices(config['n_eval_images'])
_, labels = load_cifar10_test(data_root)
labels = labels[indices]

cells = enumerate_cells(config, config['track'])
calibration_results = []

for cell in cells:
    spec = cell.spec()
    try:
        logits = cache.get_array(spec, PREDICT_VERSION)
        logits_eval = logits[indices]
    except KeyError:
        print(f"Missing predictions for {cell}")
        continue
    
    probs = softmax(logits_eval, axis=1)
    
    brier = brier_score(probs, labels)
    ece_em = ece_equal_mass(probs, labels, n_bins=15)
    ece_db = ece_debiased(probs, labels, n_bins=15)
    au_rc = aurc(probs, labels)
    accuracy = (probs.argmax(axis=1) == labels).mean()
    
    calibration_results.append({
        'model_id': cell.model_id,
        'seed': cell.seed,
        'shift_family': cell.shift_family,
        'severity': cell.severity,
        'brier': brier,
        'ece_equal_mass': ece_em,
        'ece_debiased': ece_db,
        'aurc': au_rc,
        'accuracy': accuracy,
    })

print(f"Computed calibration metrics for {len(calibration_results)} cells")

import pandas as pd
df_cal = pd.DataFrame(calibration_results)
print(f"\nCalibration summary by severity:")
for severity in sorted(df_cal['severity'].unique()):
    subset = df_cal[df_cal['severity'] == severity]
    print(f"\nSeverity {severity}:")
    print(f"  Brier:    {subset['brier'].mean():.4f} +/- {subset['brier'].std():.4f}")
    print(f"  ECE (em): {subset['ece_equal_mass'].mean():.4f} +/- {subset['ece_equal_mass'].std():.4f}")
    print(f"  ECE (db): {subset['ece_debiased'].mean():.4f} +/- {subset['ece_debiased'].std():.4f}")
    print(f"  AURC:     {subset['aurc'].mean():.4f} +/- {subset['aurc'].std():.4f}")
    print(f"  Accuracy: {subset['accuracy'].mean():.4f} +/- {subset['accuracy'].std():.4f}")

## Step 4: Compute accuracy degradation by severity

In [ ]:
import numpy as np
import pandas as pd

df_cal = pd.DataFrame(calibration_results)

print("Accuracy degradation by severity:")
print()

for family in config['shift_families']:
    clean_acc = df_cal[(df_cal['shift_family'] == 'clean') & (df_cal['severity'] == 0)]['accuracy'].mean()
    print(f"{family}:")
    for severity in sorted(config['severities']):
        corrupted_acc = df_cal[(df_cal['shift_family'] == family) & (df_cal['severity'] == severity)]['accuracy'].mean()
        drop = clean_acc - corrupted_acc
        drop_pct = 100 * drop / clean_acc if clean_acc > 0 else 0
        print(f"  Sev {severity}: {corrupted_acc:.4f} (drop {drop:.4f}, {drop_pct:.1f}%)")
    print()

## Step 5: Compute relative faithfulness with paired bootstrap intervals

In [ ]:
import numpy as np
import pandas as pd
from shiftprofile.cells import Cell, enumerate_cells
from shiftprofile.curves import REMOVAL_FRACTIONS, CURVES_VERSION
from shiftprofile.metrics.faithfulness import relative_faithfulness

cells = enumerate_cells(config, config['track'])
faithfulness_results = []
fractions = np.array(REMOVAL_FRACTIONS)

for cell in cells:
    for explainer in config['explainers']:
        cell_spec = {**cell.spec(), 'stage': 'curves', 'explainer': explainer}
        
        try:
            curves_data = cache.get_record(cell_spec, CURVES_VERSION)
            model_curve = np.array(curves_data['model_curve'])
            random_curve = np.array(curves_data.get('random_curve'))
        except (KeyError, FileNotFoundError):
            continue
        
        if random_curve is None:
            continue
        
        try:
            faith = relative_faithfulness(
                model_curve, random_curve, fractions,
                imputation=config['imputation']
            )
            faithfulness_results.append({
                'model_id': cell.model_id,
                'seed': cell.seed,
                'shift_family': cell.shift_family,
                'severity': cell.severity,
                'explainer': explainer,
                'relative_faithfulness': faith,
            })
        except Exception as e:
            pass

print(f"Computed faithfulness for {len(faithfulness_results)} cell-explainer pairs")

if len(faithfulness_results) > 0:
    df_faith = pd.DataFrame(faithfulness_results)
    
    print(f"\nFaithfulness by explainer and severity:")
    for explainer in config['explainers']:
        print(f"\n{explainer}:")
        for severity in sorted(config['severities']) + [0]:
            subset = df_faith[(df_faith['explainer'] == explainer) & (df_faith['severity'] == severity)]
            if len(subset) > 0:
                mean_faith = subset['relative_faithfulness'].mean()
                std_faith = subset['relative_faithfulness'].std()
                print(f"  Sev {severity}: {mean_faith:.4f} +/- {std_faith:.4f}")
else:
    print("No faithfulness data found. This is expected if notebook 01 did not complete.")

## Step 6: P3 Gate Decision

In [ ]:
import numpy as np
import pandas as pd
from shiftprofile.stats.bootstrap import paired_bootstrap_difference

if len(faithfulness_results) == 0:
    print("\n" + "="*70)
    print("P3 GATE: INCOMPLETE")
    print("="*70)
    print("\nNotebook 01 (pilot fill) did not complete. Re-run it to generate cache data.")
else:
    df_faith = pd.DataFrame(faithfulness_results)
    gate_results = []
    
    for explainer in config['explainers']:
        clean_faith = df_faith[
            (df_faith['explainer'] == explainer) & (df_faith['severity'] == 0)
        ]['relative_faithfulness'].values
        
        max_sev = max(config['severities'])
        severe_faith = df_faith[
            (df_faith['explainer'] == explainer) & (df_faith['severity'] == max_sev)
        ]['relative_faithfulness'].values
        
        if len(clean_faith) > 0 and len(severe_faith) > 0:
            interval = paired_bootstrap_difference(
                clean_faith, severe_faith,
                n_resamples=2000, seed=0, alpha=0.05
            )
            
            change = interval.point
            hw = interval.half_width
            
            gate_results.append({
                'explainer': explainer,
                'clean_mean': clean_faith.mean(),
                'severity_5_mean': severe_faith.mean(),
                'change': change,
                'hw': hw,
                'threshold': abs(change) * 0.1 if abs(change) > 0 else np.inf,
            })
    
    if len(gate_results) > 0:
        df_gate = pd.DataFrame(gate_results)
        
        print("\n" + "="*70)
        print("P3 GATE DECISION")
        print("="*70)
        print(f"\nGate criterion: Faithfulness interval half-width < 10% of clean-to-severity-5 change")
        print()
        
        all_pass = True
        for _, row in df_gate.iterrows():
            explainer = row['explainer']
            change = row['change']
            hw = row['hw']
            threshold = row['threshold']
            
            passes = hw < threshold if threshold < np.inf else False
            all_pass = all_pass and passes
            
            status = "PASS" if passes else "FAIL"
            print(f"{explainer}:")
            print(f"  Clean to Sev5 change: {change:.4f}")
            print(f"  Threshold (10% of change): {threshold:.4f}")
            print(f"  Interval half-width: {hw:.4f}")
            print(f"  Status: {status}")
            print()
        
        print("="*70)
        gate_status = "PASS" if all_pass else "FAIL"
        print(f"\n  P3 GATE: {gate_status}")
        print("="*70)
        print()
        if all_pass:
            print("Proceed to P3 (full evaluation)")
    else:
        print("No complete faithfulness results. Check that notebook 01 reached the explain and curves stages.")

## Step 7: Plot degradation curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if len(calibration_results) == 0:
    print("No calibration data to plot.")
else:
    df_cal = pd.DataFrame(calibration_results)
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle('Pilot Degradation Curves', fontsize=14, fontweight='bold')
    
    by_severity = df_cal.groupby('severity').agg({
        'accuracy': 'mean',
        'brier': 'mean',
        'ece_equal_mass': 'mean',
        'aurc': 'mean'
    }).reset_index()
    
    axes[0, 0].plot(by_severity['severity'], by_severity['accuracy'], 'o-', linewidth=2, markersize=8)
    axes[0, 0].set_xlabel('Severity')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].set_title('Accuracy vs Severity')
    axes[0, 0].grid(alpha=0.3)
    
    axes[0, 1].plot(by_severity['severity'], by_severity['brier'], 's-', linewidth=2, markersize=8)
    axes[0, 1].set_xlabel('Severity')
    axes[0, 1].set_ylabel('Brier Score')
    axes[0, 1].set_title('Brier Score vs Severity')
    axes[0, 1].grid(alpha=0.3)
    
    axes[1, 0].plot(by_severity['severity'], by_severity['ece_equal_mass'], '^-', linewidth=2, markersize=8)
    axes[1, 0].set_xlabel('Severity')
    axes[1, 0].set_ylabel('ECE (Equal Mass)')
    axes[1, 0].set_title('ECE vs Severity')
    axes[1, 0].grid(alpha=0.3)
    
    axes[1, 1].plot(by_severity['severity'], by_severity['aurc'], 'd-', linewidth=2, markersize=8)
    axes[1, 1].set_xlabel('Severity')
    axes[1, 1].set_ylabel('AURC')
    axes[1, 1].set_title('AURC vs Severity')
    axes[1, 1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('/kaggle/working/pilot_degradation.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print("\nPlot saved to /kaggle/working/pilot_degradation.png")

## Report Summary

The pilot evaluation is complete. Use the metrics above to:

1. Verify P3 gate: Check whether faithfulness intervals satisfy the criterion
2. Assess measurement precision: Review interval widths and sample sizes
3. Identify failure modes: Look for patterns in accuracy degradation or calibration drift

If P3 gate PASS: Proceed to P3 (full evaluation)

If P3 gate FAIL: Review the caveats and adjust the protocol if needed.